# Chapter 6 — Testing
### ML Engineer (Production) Interview Playbook

**Topics:** pytest · Fixtures · Mock · Unit/Integration · Async · Coverage · ML Testing · CI

> Writing tests is a mark of a mature engineer and essential for a production role. Tests catch bugs
> early, act as living documentation of the code's behavior, and let you refactor without fear. A tech
> lead usually doesn't ask "do you write tests?" but rather "how do you write testable code, and what
> do you test, and how?" This chapter covers the tooling (pytest) as well as the philosophy and the
> specifics of testing ML code.

### The Test Pyramid

A balanced guide for the test mix: the base of the pyramid is a large number of fast, cheap unit tests;
the middle layer is a smaller number of integration tests; and the tip is a small number of slow,
brittle end-to-end tests. Inverting this pyramid (lots of E2E tests) leads to a slow, unstable suite.

**Interview tip:** Mindset: the goal of testing isn't "proving correctness" (often impossible), but
"increasing confidence" at a reasonable cost. Fast tests focused on critical paths give the most
value.

## 6.1 pytest Basics and Parametrize

pytest is popular for its simplicity: `test_*` functions and plain `assert` (with rich introspection
that shows the actual and expected values). No need for `assertEqual`-style methods.

In [ ]:
import pytest

def add(a, b): return a + b

def test_add():
    assert add(2, 3) == 5   # on failure, pytest shows the values

# Parametrizing: one test, many cases
@pytest.mark.parametrize("a, b, expected", [
    (1, 1, 2),
    (0, 0, 0),
    (-1, 1, 0),
])
def test_add_cases(a, b, expected):
    assert add(a, b) == expected

# Testing a raised exception
def test_divide_by_zero():
    with pytest.raises(ZeroDivisionError):
        1 / 0


**Interview tip:** Useful markers: `@pytest.mark.skip` (skip), `@pytest.mark.xfail` (expected to
fail), and custom markers for categorization (e.g. `@pytest.mark.slow` so they run separately in CI).

## 6.2 Fixtures and `conftest`

A fixture injects ready-made data or a resource (a test database, a client, sample data) into tests and
cleans up afterward. With `yield`, the code before it is setup and the code after it is teardown.

In [ ]:
import pytest

@pytest.fixture
def sample_user():
    return User(id="1", country="NL")

@pytest.fixture
def db_session():
    session = create_test_session()
    yield session       # the test runs here
    session.rollback()  # teardown: cleanup
    session.close()

def test_save(db_session, sample_user):   # injected by name
    db_session.add(sample_user)
    assert db_session.get(User, "1") is not None


### Scope and `conftest.py`

| scope | fixture lifetime |
|---|---|
| function (default) | Created fresh for every test function |
| class | Once per test class |
| module | Once per file |
| session | Once for the entire test run |

You put shared fixtures in `conftest.py` so they're available in every test in that directory without
an import. For expensive resources (like starting a test database container), use a wider scope
(`session`) so it's only created once.

### Q6.2 — How do you ensure each test is independent of the others?

A few principles: (1) Each test gets its own fresh state from a fixture (function scope) and cleans up
in teardown. (2) I avoid shared, global state; if needed, I reset it in the fixture. (3) For a database,
I run each test in a transaction and roll it back at the end so nothing persists. (4) Tests shouldn't
depend on execution order; they must pass in any order. Test independence is a prerequisite for
parallel execution and stability.

## 6.3 Mock, Patch, and Test Doubles

Mocking replaces slow or external dependencies (an API, database, model, the clock) with a fake version
so tests stay fast, deterministic, and isolated. "Test double" is an umbrella term with several kinds,
and knowing the difference is often asked:

| Type | What it does |
|---|---|
| Dummy | Just fills a parameter; never actually used |
| Stub | Returns a predetermined response |
| Spy | Like a stub, but also records calls |
| Mock | Also verifies expected behavior (which method, with what arguments) |
| Fake | A lightweight real implementation (e.g. an in-memory database) |

In [ ]:
from unittest.mock import Mock, patch

def test_fraud_service():
    repo = Mock()
    repo.get.return_value = User(id="1", country="NL")   # stub
    model = Mock()
    model.predict.return_value = 0.9

    service = FraudService(repo, model, cache=Mock())
    result = service.evaluate("1", 100.0)

    assert result == 0.9
    repo.get.assert_called_once_with("1")   # behavior verification (mock)

# side_effect to simulate an error or a sequence of values
def test_retry():
    api = Mock()
    api.call.side_effect = [TimeoutError, TimeoutError, "ok"]


**Note:** The most important mock trap: "where to patch." You must patch the object where it's
*used*, not where it's *defined*. If module A imported it with `from mod import func`, you must patch
`A.func`, not `mod.func`. This is the most common mocking mistake.

Also use `spec`/`autospec` so the mock respects the object's real signature — otherwise a mock accepts
any method call, and a green test can hide a real bug.

### Q6.3 — What's the difference between Mock and Stub?

Both stand in for a real dependency, but their purpose differs. A stub just returns predetermined data
so the test can proceed (state-based: we check the output). A mock also verifies behavioral
expectations: that a specific method was called with the right arguments, and how many times
(interaction-based). Rule of thumb: if you only care about the output, a stub is enough; if you want to
verify the code interacted correctly with a dependency (e.g. it actually sent an email), use a mock and
`assert_called`. Over-mocking behavior makes tests brittle and implementation-dependent.

## 6.4 Unit vs. Integration Tests

| Feature | Unit Test | Integration Test |
|---|---|---|
| Scope | One isolated unit of logic | Several real components together |
| Dependencies | Mocked | Real (e.g. a test database) |
| Speed | Very fast | Slower |
| What it catches | Internal logic bugs | Bugs in component interaction and configuration |

Both are needed and complementary: unit tests check logic fast but don't tell you whether components
actually work together; integration tests fill that gap (e.g. whether a real query against a real
database is correct). This is exactly what the test pyramid's balance means: a large base of unit
tests, a thinner layer of integration.

### Q6.4 — You have code that calls an external API. How do you test it?

Two levels of testing: (1) In a unit test, I mock the external API call to isolate and deterministically
check the logic around it (building the request, processing the response, error handling and retry)
without depending on the network or a real service. For error scenarios (timeout, 500), I use
`side_effect`. (2) In a limited integration layer, I test against a sandbox version or a mock server
(e.g. a fake HTTP server) to make sure I correctly understand the real contract. I never depend on the
real production API in automated tests, since that becomes slow, costly, and unstable.

## 6.5 Testing Async Code and Testing FastAPI

Testing async code needs a runner (like `pytest-asyncio`). For testing FastAPI endpoints, we use
`TestClient` or `httpx.AsyncClient`, and most importantly, we swap dependencies via
`dependency_overrides`.

In [ ]:
import pytest
from fastapi.testclient import TestClient
from app.main import app
from app.deps import get_db

# Replace the real database with a test version
def override_get_db():
    return TestSession()

app.dependency_overrides[get_db] = override_get_db
client = TestClient(app)

def test_score_endpoint():
    resp = client.post("/score", json={"user_id": "1", "amount": 10})
    assert resp.status_code == 200
    assert 0 <= resp.json()["risk"] <= 1

# Testing a coroutine directly
@pytest.mark.asyncio
async def test_service():
    result = await service.evaluate("1", 100)
    assert result is not None


**Interview tip:** This is exactly where the power of DI from the Backend chapter pays off:
because the endpoint gets the database from `Depends`, the test version is injected without changing
the app's code. If business logic had been tangled into the controller, this simple test wouldn't have
been possible.

## 6.6 Coverage

Test coverage is the percentage of code executed while running the tests. Tools: `coverage.py` or
`pytest-cov`. Two important kinds: line coverage (was the line executed) and branch coverage (was each
path of an if/else executed — more precise).

In [ ]:
pytest --cov=app --cov-report=term-missing --cov-branch

# term-missing shows the lines that aren't covered


**Note:** High coverage isn't high quality. A line can execute without its behavior being
asserted (execution ≠ verification). Also, blindly targeting 100% is an anti-pattern: it leads to
writing worthless tests just to bump the number. Coverage is a tool for finding untested code, not a
quality metric.

### Q6.6 — How much coverage is enough?

The absolute number doesn't matter; what matters is covering critical paths and edge cases. A payment
module with 80% coverage that tests every error path and boundary is better than a module with 100%
coverage that only exercises the happy path (with no meaningful assertions). I use coverage to find
important untested spots, not as a numeric target. I prefer branch coverage over line coverage because
it measures decision branches. And I'm careful that high coverage doesn't give a false sense of
security.

## 6.7 Testing ML Code and Data

Testing ML systems goes beyond ordinary software testing because behavior also depends on data, not
just code. Three levels of testing come up:

- **Data tests:** schema validation, value ranges, unexpected `NULL`s, and input distribution (with
  tools like pandera or Great Expectations). Bad data produces a bad model.
- **Code tests (feature/pipeline):** unit-test every feature-engineering and pipeline step like normal
  code; especially feature-generation consistency between train and serve.
- **Model tests (behavioral):** instead of only checking an overall metric, test the model's behavior:
  invariance tests (an irrelevant change shouldn't change the output), directional tests (increasing a
  feature should move the output in the expected direction), and minimum-functionality tests on simple
  cases.

In [ ]:
import pytest

# Determinism test: with a fixed seed, output should be reproducible
def test_deterministic():
    set_seed(42)
    a = train_and_predict(data)
    set_seed(42)
    b = train_and_predict(data)
    assert a == pytest.approx(b)

# Directional test: increasing the transaction amount shouldn't decrease risk
def test_directional():
    low = score({"amount": 10})
    high = score({"amount": 100000})
    assert high >= low

# Data test: a constraint on the input
def test_no_negative_amounts(df):
    assert (df["amount"] >= 0).all()


**Interview tip:** The non-determinism challenge: many ML algorithms are random. For testing,
fix the seed and use a tolerance (`pytest.approx`) instead of exact equality. Make non-deterministic
sources like time and randomness injectable so you can control them in tests.

### Q6.7 — How do you test a function whose output is random/non-deterministic?

A few techniques: (1) I make the source of randomness controllable — fixing the seed or injecting the
random generator so the output is reproducible. (2) Instead of exact equality on floating-point
numbers, I use a tolerance (`pytest.approx`). (3) If the exact output isn't predictable, I test
properties: e.g. the output falls in `[0,1]`, or a directional relationship holds, or the output's
distribution has the expected statistical property. (4) I mock non-deterministic external sources
(time, network). The goal is to make the test deterministic without destroying the algorithm's random
nature.

## 6.8 Best Practices and CI

The FIRST principles for good tests: **F**ast, **I**solated, **R**epeatable (deterministic),
**S**elf-validating (determines pass/fail itself), **T**imely (written at the right time). Tests run in
CI on every push and should fail the build when they fail.

### Flaky tests

A flaky test sometimes passes and sometimes fails with no code change; it destroys trust in the whole
suite. Common causes and fixes:

- **Dependency on time/date:** inject or mock the clock (instead of the real `datetime.now`).
- **Dependency on execution order or shared state:** isolate each test and reset state.
- **Network/external service:** mock it; don't depend on the real service.
- **Concurrency and races:** control shared resources and make waits deterministic.

**Note:** Never "fix" a flaky test by "retrying a few times until it passes" — this hides the problem.
Find and fix the root cause of instability, or rewrite the test to be deterministic.

### Q6.8 — What makes a test flaky, and how do you fix it?

A flaky test sometimes passes and sometimes fails with no code change. Main causes: dependency on real
time, dependency on test execution order or shared state, reliance on the network or an external
service, and race conditions in concurrent code. Fixes: make non-deterministic sources (time, random)
injectable and fixed in tests; fully isolate each test and reset state in setup/teardown; mock external
dependencies; and for async code, make waits deterministic instead of relying on `sleep`. Flaky tests
must be taken seriously because they destroy trust in the entire suite.

## Quick Chapter Summary (Cheat Sheet)

Review this on interview day:

- **Test pyramid:** lots of unit, less integration, a few E2E; the goal is increasing confidence at a
  reasonable cost.
- **pytest:** plain `assert`, `parametrize` for multiple cases, `pytest.raises` for exceptions,
  markers.
- **Fixtures:** `yield` for setup/teardown; scope (`function`..`session`); `conftest` for sharing;
  test independence.
- **Mock:** a stub returns output, a mock verifies behavior; "patch where it's used"; `spec` for a
  correct signature.
- **Unit/Integration:** unit is isolated and fast (mock), integration uses real components; mock
  external APIs.
- **async/FastAPI:** `pytest-asyncio`; `TestClient`; `dependency_overrides` to inject the test version.
- **Coverage:** execution ≠ verification; branch > line; critical paths matter more than hitting 100%.
- **ML testing:** data tests (schema/range), behavioral model tests (invariance/directional),
  determinism via seed and `pytest.approx`.
- **CI:** FIRST principles; fix flaky tests at the root, not with retries; fail the build in CI on
  failure.